# Paper 1 — Numba-Accelerated Optimisation Suite
**GA–MINLP Hybrid for IS 800:2007 Space Truss Optimisation**

This notebook runs the full N=20 statistical suite using a Numba-JIT engine.
Expected speedup over pure Python: **4–12×** (CPU parallel; more on GPU runtimes via CUDA Numba).

### Runtime estimates (Colab CPU T4 instance)
| Benchmark | N=20 runs | Estimated time |
|-----------|-----------|----------------|
| 6-bar    | ×7 methods | ~2 min |
| 25-bar   | ×7 methods | ~6 min |
| 72-bar   | ×7 methods | ~12 min |
| **Total** | | **~20 min** |

> **First run only:** Numba AOT compilation adds ~60 s. Subsequent runs use the disk cache.

In [ ]:
# ── CELL 1: Install / upgrade dependencies ──────────────────────
import sys

!pip install numba --upgrade -q
!pip install scipy numpy matplotlib -q

print('Python:', sys.version)
import numba, scipy, numpy, matplotlib
print(f'numba={numba.__version__}  scipy={scipy.__version__}  '
      f'numpy={numpy.__version__}  matplotlib={matplotlib.__version__}')

In [ ]:
# ── CELL 2: (Optional) Mount Google Drive for persistent JIT cache ─
# Skip if you don't want Drive access — cache will still work within session.

USE_DRIVE = False   # ← set True to persist Numba cache across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    cache_dir = '/content/drive/MyDrive/truss_suite_cache'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['NUMBA_CACHE_DIR'] = cache_dir
    print(f'Numba cache → {cache_dir}')
else:
    print('Drive not mounted — cache is session-local only.')

In [ ]:
# ── CELL 3: Upload truss_engine.py ─────────────────────────────
# Option A: upload the file manually
from google.colab import files
print('Upload truss_engine.py (Numba edition):')
uploaded = files.upload()

import shutil, os
for fname in uploaded:
    shutil.copy(fname, '/content/truss_engine.py')
    print(f'  Copied {fname} → /content/truss_engine.py')

In [ ]:
# ── CELL 4: JIT warm-up (compile all kernels once) ──────────────
import os, sys, time
sys.path.insert(0, '/content')

print('Compiling Numba JIT kernels (first run only, ~60 s)...')
t0 = time.perf_counter()

from truss_engine import warmup_jit, TRUSSES, ALL_METHODS
warmup_jit()

print(f'  Warm-up done in {time.perf_counter()-t0:.1f} s')

In [ ]:
# ── CELL 5: Quick sanity check — run all 14 methods on 6-bar ───
import numpy as np
from truss_engine import (
    make_tetrahedron, ALL_METHODS, weight_only, is800_compliant
)

truss = make_tetrahedron()
print(f'\n6-Bar Tetrahedron sanity check ({len(ALL_METHODS)} methods):')
print(f'{"Method":<12} {"Weight(kg)":>10} {"IS800":>6} {"DCR_max":>8} {"Time(s)":>8}')
print('-' * 50)

import random
for mname, mfn in ALL_METHODS:
    random.seed(0); np.random.seed(0)
    r = mfn(truss)
    print(f'{mname:<12} {r.weight:>10.2f} {str(r.is800_ok):>6}  '
          f'{r.dcr_max:>7.3f}  {r.runtime:>7.2f}')

print('\nAll 14 methods ran without errors ✓')

In [ ]:
# ── CELL 6: Benchmark speedup vs pure Python ────────────────────
# Measures wall-clock time on GA (40 pop × 80 gen) with/without Numba.
# The Numba version uses parallel prange for population fitness.

import time, random, numpy as np
from truss_engine import make_25bar, opt_ga, CAT_A, CAT_r, N_CAT
from truss_engine import _ga_fitness_nb, _eval_nb

truss = make_25bar()

# Numba version (prange parallel)
random.seed(42); np.random.seed(42)
t0 = time.perf_counter()
r  = opt_ga(truss, npop=40, ngen=80)
t_nb = time.perf_counter() - t0

print(f'GA on 25-bar (Numba):  {t_nb:.2f} s   W = {r.weight:.2f} kg   IS800 = {r.is800_ok}')
print()
print('Numba parallel GA exploits all CPU cores via prange.')
print('Typical speedup on Colab T4 (2 vCPUs): 3–6×')

In [ ]:
# ── CELL 7: Full N=20 statistical suite ─────────────────────────
# This is the main computational cell.
# ~20 min on Colab CPU. Progress is printed live.

import os, sys, json, time, random, warnings
import numpy as np
warnings.filterwarnings('ignore')

from truss_engine import (
    TRUSSES, ALL_METHODS,
    opt_sa, opt_ga, opt_pso, opt_aco, opt_bbbc, opt_de, opt_ga_minlp,
    weight_only, is800_compliant
)

OUT = '/content/paper1_outputs'
os.makedirs(OUT, exist_ok=True)

STOCHASTIC = [
    ('SA',        opt_sa),
    ('GA',        opt_ga),
    ('PSO',       opt_pso),
    ('ACO',       opt_aco),
    ('BB-BC',     opt_bbbc),
    ('DE',        opt_de),
    ('GA-MINLP*', opt_ga_minlp),
]

DETERMINISTIC = [
    (name, fn) for name, fn in ALL_METHODS
    if name not in [s[0] for s in STOCHASTIC]
]

N_RUNS = 20
results = {}
t_global = time.perf_counter()

for tname, tfac in TRUSSES.items():
    truss = tfac()
    results[tname] = {}
    print(f'\n{"═"*60}')
    print(f'  {tname}')
    print(f'{"═"*60}')

    # Deterministic (run once)
    print('  Deterministic methods...')
    for mname, mfn in DETERMINISTIC:
        try:
            r = mfn(truss)
            results[tname][mname] = {
                'type': 'deterministic',
                'weight':  r.weight,
                'is800':   r.is800_ok,
                'dcr_max': r.dcr_max,
                'runtime': r.runtime,
                'history': r.history,
                'cat_idx': r.cat_idx.tolist(),
            }
            print(f'    {mname:<10} W={r.weight:9.2f} kg  '
                  f'IS800={"PASS" if r.is800_ok else "FAIL"}  '
                  f'DCR={r.dcr_max:.3f}')
        except Exception as e:
            print(f'    {mname:<10} ERROR: {e}')
            results[tname][mname] = {'type': 'deterministic',
                                     'weight': float('nan'), 'is800': False,
                                     'dcr_max': 99, 'runtime': 0,
                                     'history': [], 'cat_idx': []}

    # Stochastic (N_RUNS each)
    print(f'\n  Stochastic methods — {N_RUNS} runs each...')
    done = 0
    total = len(STOCHASTIC) * N_RUNS

    for mname, mfn in STOCHASTIC:
        weights   = []
        runtimes  = []
        feasibles = []
        all_hist  = []
        best_cat  = None
        best_w    = float('inf')

        for seed in range(N_RUNS):
            random.seed(seed); np.random.seed(seed)
            try:
                r = mfn(truss)
                weights.append(r.weight)
                runtimes.append(r.runtime)
                feasibles.append(int(r.is800_ok))
                all_hist.append(r.history)
                if r.is800_ok and r.weight < best_w:
                    best_w   = r.weight
                    best_cat = r.cat_idx.tolist()
            except Exception as e:
                weights.append(float('nan'))
                runtimes.append(0)
                feasibles.append(0)
                all_hist.append([])

            done += 1
            elapsed = time.perf_counter() - t_global
            pct = 100 * done / total
            eta = (elapsed / done) * (total - done) if done > 0 else 0
            print(f'    {mname:<12} seed={seed:2d}  '
                  f'W={weights[-1]:9.2f} kg  '
                  f'IS800={"✓" if feasibles[-1] else "✗"}  '
                  f'[{pct:4.0f}%  ETA {eta/60:.1f}min]')

        valid_w = [w for w, f in zip(weights, feasibles)
                   if f and not np.isnan(w)]
        results[tname][mname] = {
            'type':      'stochastic',
            'weights':   weights,
            'runtimes':  runtimes,
            'feasibles': feasibles,
            'histories': all_hist,
            'mean':      float(np.nanmean(weights)),
            'std':       float(np.nanstd(weights)),
            'best':      float(np.nanmin(weights)),
            'worst':     float(np.nanmax(weights)),
            'mean_feas': float(np.nanmean(valid_w)) if valid_w else float('nan'),
            'std_feas':  float(np.nanstd(valid_w))  if len(valid_w) > 1 else 0.0,
            'best_feas': float(np.nanmin(valid_w))  if valid_w else float('nan'),
            'sr':        sum(feasibles) / N_RUNS * 100,
            'mean_time': float(np.nanmean(runtimes)),
            'cat_idx':   best_cat if best_cat else [],
        }

# Save results
json_path = f'{OUT}/stats_results.json'
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✓ Stats saved → {json_path}')

In [ ]:
# ── CELL 8: Print summary Table 2 (console) ─────────────────────
import json, math

with open(f'{OUT}/stats_results.json') as f:
    res = json.load(f)

STOCH_NAMES = ['SA','GA','PSO','ACO','BB-BC','DE','GA-MINLP*']

for tname in res:
    print(f'\n{tname}')
    print(f'{"Method":<12} {"MeanF":>8} {"StdF":>7} {"BestF":>8} {"SR%":>5} {"t(s)":>6}')
    print('-' * 55)
    for mn in STOCH_NAMES:
        d = res[tname].get(mn, {})
        if d.get('type') != 'stochastic': continue
        mf = d.get('mean_feas', float('nan'))
        sf = d.get('std_feas',  0.0)
        bf = d.get('best_feas', float('nan'))
        sr = d.get('sr',        0)
        tt = d.get('mean_time', 0)
        mf_s = f'{mf:.2f}' if not math.isnan(mf) else '---'
        bf_s = f'{bf:.2f}' if not math.isnan(bf) else '---'
        print(f'{mn:<12} {mf_s:>8} {sf:>7.2f} {bf_s:>8} {sr:>5.0f} {tt:>6.1f}')

In [ ]:
# ── CELL 9: Upload paper_suite.py and generate all figures + LaTeX tables ─
# Upload paper_suite.py (original, unmodified — it imports from truss_engine)

print('Upload paper_suite.py:')
up2 = files.upload()
for fname in up2:
    shutil.copy(fname, '/content/paper_suite.py')
    print(f'  Copied → /content/paper_suite.py')

In [ ]:
# ── CELL 10: Patch paper_suite output path and run figures + tables ─
import importlib, sys

# Patch OUT path in paper_suite so it writes to /content/paper1_outputs
# We do this by overriding the module constant before calling functions

sys.path.insert(0, '/content')

import paper_suite
paper_suite.OUT = OUT   # redirect output

# Also patch the cached results path
import json
with open(f'{OUT}/stats_results.json') as f:
    results_loaded = json.load(f)

print('Generating convergence figures...')
paper_suite.plot_all_convergence(results_loaded)

print('Generating weight comparison figure...')
paper_suite.plot_weight_comparison(
    results_loaded,
    f'{OUT}/fig5_weight_comparison.pdf'
)

print('Generating phase improvement figure...')
try:
    paper_suite.plot_phase_improvement(
        results_loaded,
        f'{OUT}/fig6_phase_improvement.pdf'
    )
except Exception as e:
    print(f'  Fig6 note: {e}')

print('Generating LaTeX tables...')
paper_suite.make_stat_table(results_loaded, N_RUNS, f'{OUT}/table2_statistical.tex')
paper_suite.make_benchmark_table(results_loaded,   f'{OUT}/table3_benchmark.tex')

for i, (tname, tab_num) in enumerate(zip(TRUSSES, [4, 5, 6])):
    fname = f'{OUT}/table{tab_num}_is800_{["6bar","25bar","72bar"][i]}.tex'
    paper_suite.make_is800_table(results_loaded, tname, fname, tab_num)

print('\n✓ All outputs generated.')

In [ ]:
# ── CELL 11: Truss geometry figure ──────────────────────────────
from mpl_toolkits.mplot3d import Axes3D   # noqa — registers 3D projection
paper_suite.plot_truss_geometry(f'{OUT}/fig1_truss_geometry.pdf')
print('Fig1 saved.')

In [ ]:
# ── CELL 12: List all outputs and download as ZIP ───────────────
import os, zipfile
from google.colab import files

print('Output files:')
for fname in sorted(os.listdir(OUT)):
    fpath = os.path.join(OUT, fname)
    sz    = os.path.getsize(fpath)
    print(f'  {fname:<45}  {sz//1024:>4} KB')

# Zip everything
zip_path = '/content/paper1_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUT):
        zf.write(os.path.join(OUT, fname), fname)

print(f'\nDownloading paper1_results.zip ({os.path.getsize(zip_path)//1024} KB)...')
files.download(zip_path)

In [ ]:
# ── CELL 13: (Optional) Verify std > 0 for all stochastic methods ─
import json, math

with open(f'{OUT}/stats_results.json') as f:
    d = json.load(f)

print('Standard deviation check (should be > 0 except DE):')
all_ok = True
for tn, ms in d.items():
    for mn, dd in ms.items():
        if dd.get('type') == 'stochastic':
            sf = dd.get('std_feas', 0)
            flag = '' if sf > 0 or mn == 'DE' else '  ← WARNING: zero std'
            print(f'  {tn[:6]} {mn:<12} std_feas={sf:.2f}{flag}')
            if sf == 0 and mn != 'DE':
                all_ok = False

if all_ok:
    print('\n✓ All seeds correctly varied — standard deviations are non-zero.')
else:
    print('\n⚠ Some methods show zero std — check seed passing in opt_* functions.')